# poetru training notebook

Poetru-75M is a Russian poetry causal language model trained on IlyaGusev/stihi_ru. The sections below state the mathematical formulation and then run the full pipeline.

## Chinchilla budget

The corpus contains about $4.55 \times 10^8$ BPE tokens at 2 GB. Training for 3 epochs exposes the model to roughly $1.37 \times 10^9$ token positions on prefixes truncated to 512 BPE tokens. Chinchilla places compute-optimal models near 20 training tokens per parameter, which yields $N_{\ast} \approx 6.8 \times 10^7$. The default `TransformerConfig` uses about $7.5 \times 10^7$ trainable parameters.

## RoPE

For hidden size $d=640$, heads $H=8$, and head dimension $d_h=80$, RoPE uses frequencies

$$\theta_k = 10000^{-2k/d_h}, \quad k=0,\ldots,d_h/2-1$$

At sequence position $m$ the plane spanned by coordinates $(2k,2k+1)$ rotates by angle $m\theta_k$:

$$\begin{pmatrix} q'_{2k} \\ q'_{2k+1} \end{pmatrix}=
\begin{pmatrix} \cos m\theta_k & -\sin m\theta_k \\ \sin m\theta_k & \cos m\theta_k \end{pmatrix}
\begin{pmatrix} q_{2k} \\ q_{2k+1} \end{pmatrix}$$

Keys receive the same transform before attention scores $QK^\top/\sqrt{d_h}$.

## MLA and GQA

Hidden states $h_t \in \mathbb{R}^{640}$ produce queries with eight heads. Keys and values pass through latent compression $c_t = W_d h_t \in \mathbb{R}^{640}$, then expand to four KV heads with matrices $W_k, W_v \in \mathbb{R}^{(4 d_h) \times 640}$. Each KV head is repeated twice so every query head attends to a KV head.

## Watermark

Following Kirchenbauer et al., 2023, token $s_t$ is sampled after adding bias $\delta$ to logits indexed by a green list $G_t$ of size $\gamma |\mathcal{V}|$. The list is a deterministic function of the previous token $s_{t-1}$. Detection counts green hits and applies a z-test against mean $\gamma T$.

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from main import train_tokenizer, train_model, generate_poems, evaluate_perplexity, evaluate_watermark, author_pca, publish_hub

train_tokenizer(ROOT)
train_model(ROOT)
generate_poems(ROOT)
evaluate_perplexity(ROOT)
evaluate_watermark(ROOT)
author_pca(ROOT)
publish_hub(ROOT)

## Inference

Watermarked continuation from a trained checkpoint.

In [ ]:
import sys
import torch
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from hub_utils import download_inference_artifacts
from bpe_tokenizer import ByteBPETokenizerWrapper
from checkpoint_utils import load_checkpoint
from configs import GenerationConfig
from trainer import generate_poem

download_inference_artifacts(ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = ByteBPETokenizerWrapper.from_file(ROOT / 'artifacts/tokenizer/tokenizer.json')
model, _ = load_checkpoint(ROOT / 'artifacts/checkpoints/final.pt', device)
model.eval()

prompt = 'В тишине ночной'
prompt_ids = tokenizer.encode(prompt, add_eos=False)
token_ids, _ = generate_poem(
    model,
    prompt_ids,
    eos_id=tokenizer.eos_id,
    gen_cfg=GenerationConfig(),
    device=device,
    apply_watermark=True,
)
print(tokenizer.decode(token_ids))